# DQ 检查 — ODS 层（Rule 0a/0b）

在 DWD 之前拦截两类问题：
- **Rule 0a**：每个来源的 ODS 分区行数 > 0（空文件/文件未到检测）
- **Rule 0b**：各月行数与其他月均值偏差 < 50%（异常波动检测）

任何 FAIL 将抛出异常，阻断 DWD 构建。

## 1. SparkSession

In [1]:
import os
from datetime import datetime
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = (
    SparkSession.builder
    .appName("TLC-DQ-AfterODS")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.driver.memory", "4g")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

BASE    = os.path.expanduser("~/tlc-spark-pipeline")
ODS_Y   = f"{BASE}/data/warehouse/ods/yellow"
ODS_G   = f"{BASE}/data/warehouse/ods/green"
ODS_FHV = f"{BASE}/data/warehouse/ods/fhv_hv"

TARGET_DATE = os.environ.get("TARGET_DATE", "2024-03-15")
yr  = int(TARGET_DATE.split("-")[0])
mo  = int(TARGET_DATE.split("-")[1])
day = int(TARGET_DATE.split("-")[2])
print(f"TARGET_DATE={TARGET_DATE}")

print(f"Spark {spark.version} ready | TARGET_DATE={TARGET_DATE}")


26/10/19 14:55:35 WARN Utils: Your hostname, jelly-macbook.local resolves to a loopback address: 127.0.0.1; using 192.168.1.229 instead (on interface en0)
26/10/19 14:55:35 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


26/10/19 14:55:36 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


TARGET_DATE=2024-03-15
Spark 3.5.1 ready | TARGET_DATE=2024-03-15


## 2. DQ 框架

In [2]:
results = []

def check(rule_id, layer, rule_type, description, passed, detail=""):
    status = "PASS" if passed else "FAIL"
    results.append({"rule_id": rule_id, "status": status, "description": description})
    mark = "✅" if passed else "❌"
    print(f"{mark} Rule {rule_id} [{layer}] {description}  {detail}")


## 3. Rule 0a：各来源分区行数非空

In [3]:
sources = [
    ("yellow", ODS_Y),
    ("green",  ODS_G),
    ("fhv_hv", ODS_FHV),
]

source_counts = {}  # source -> {(year, month) -> count}

for src, path in sources:
    df = spark.read.parquet(path).filter(
        (F.col("pickup_year") == yr) & (F.col("pickup_month") == mo) & (F.col("pickup_day") == day))
    monthly = (
        df.groupBy("pickup_year", "pickup_month")
          .count()
          .orderBy("pickup_year", "pickup_month")
          .collect()
    )
    all_counts = {(r.pickup_year, r.pickup_month): r["count"] for r in monthly}
    total = sum(all_counts.values())
    source_counts[src] = all_counts
    check(
        f"0a-{src}", "ODS", "non_empty",
        f"{src} ODS non-empty",
        passed=(total > 0),
        detail=f"total={total:,}  partitions={len(monthly)}"
    )


26/10/19 14:55:47 WARN GarbageCollectionMetrics: To enable non-built-in garbage collector(s) List(G1 Concurrent GC), users should configure it(them) to spark.eventLog.gcMetrics.youngGenerationGarbageCollectors or spark.eventLog.gcMetrics.oldGenerationGarbageCollectors


✅ Rule 0a-yellow [ODS] yellow ODS non-empty  total=128,813  partitions=1


✅ Rule 0a-green [ODS] green ODS non-empty  total=2,036  partitions=1


✅ Rule 0a-fhv_hv [ODS] fhv_hv ODS non-empty  total=740,231  partitions=1


## 4. Rule 0b：月度行数波动检测（偏差 < 50%）

In [4]:
for src, monthly in source_counts.items():
    counts = list(monthly.values())
    if len(counts) < 2:
        print(f"⚠️  {src}: only 1 partition, skipping volatility check")
        continue
    avg = sum(counts) / len(counts)
    for (yr, mo), cnt in monthly.items():
        deviation = abs(cnt - avg) / avg
        check(
            f"0b-{src}-{yr}{mo:02d}", "ODS", "row_volatility",
            f"{src} {yr}-{mo:02d} row count within 50% of monthly avg",
            passed=(deviation < 0.50),
            detail=f"count={cnt:,}  avg={avg:,.0f}  deviation={deviation:.1%}"
        )


⚠️  yellow: only 1 partition, skipping volatility check
⚠️  green: only 1 partition, skipping volatility check
⚠️  fhv_hv: only 1 partition, skipping volatility check


## 5. 报告 + 阻断判断

In [5]:
fail_rules = [r for r in results if r['status'] == 'FAIL']
pass_rules = [r for r in results if r['status'] == 'PASS']

print(f"\n{'='*50}")
print(f"DQ Report (ODS)  {datetime.now().strftime('%Y-%m-%d %H:%M')}")
print(f"Rules: {len(results)} total | {len(pass_rules)} PASS | {len(fail_rules)} FAIL")
print(f"={'='*49}")

spark.stop()

if fail_rules:
    for r in fail_rules:
        print(f"  ❌ {r['rule_id']}: {r['description']}")
    raise ValueError(
        f"DQ FAILED: {len(fail_rules)} ODS rule(s). "
        "Pipeline halted — check source data before DWD build."
    )
else:
    print("All ODS rules passed — proceeding to DWD.")

print("DQ after-ODS check complete.")



DQ Report (ODS)  2026-10-19 14:55
Rules: 3 total | 3 PASS | 0 FAIL


All ODS rules passed — proceeding to DWD.
DQ after-ODS check complete.
